### Loading review text
The Olist review dataset includes free-text comments in Portuguese. We haven't 
used this text anywhere yet — everything so far has been structured/numeric 
fields. Checking how much usable text we actually have before running anything 
expensive on it.

In [4]:
import pandas as pd
import sys
sys.path.append("../src")
from load_data import load_raw_tables

tables = load_raw_tables()
reviews = tables["order_reviews"]

print(f"Total reviews: {len(reviews):,}")
print(f"Reviews with comment text: {reviews['review_comment_message'].notna().sum():,}")
print(f"\nSample comments:")
print(reviews["review_comment_message"].dropna().sample(5, random_state=42).tolist())

2
Total reviews: 99,224
Reviews with comment text: 40,977

Sample comments:
['Um produto como uma carteira somente poderia ser avaliado depois de algum tempo de uso, No entanto, o produto é bem acabado e de couro conforme o especificado pela empresa.', 'Entrega no prazo , bom produto .', 'entrega rápida', 'Chegou no prazo e atendeu as minhas espectativas', 'Logística ótima entregue antes do prazo previsto, estou satisfeito. ']


### Scoping to first-order reviews only
Same rule as feature engineering — we can only use information available from a 
customer's first order, not reviews from later orders. Joining review text back 
to the first-order feature table and checking how many usable comments remain.

In [5]:
model_data = pd.read_csv("../data/processed/model_ready.csv")
first_order_features = pd.read_csv("../data/processed/first_order_features.csv")

# We need the order_id to join reviews, but it wasn't kept in first_order_features.
# Quick fix: rebuild the customer -> first order_id mapping
orders = tables["orders"]
customers = tables["customers"]

orders_customers = orders.merge(customers, on="customer_id")
orders_customers = orders_customers[orders_customers["order_status"] == "delivered"]
orders_customers["order_purchase_timestamp"] = pd.to_datetime(orders_customers["order_purchase_timestamp"])

first_orders = (
    orders_customers.sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id")
    .first()
    .reset_index()[["customer_unique_id", "order_id"]]
)

# Join review text onto first orders
first_order_reviews = first_orders.merge(
    reviews[["order_id", "review_comment_message"]], on="order_id", how="left"
)

n_with_text = first_order_reviews["review_comment_message"].notna().sum()
print(f"First orders with review text: {n_with_text:,} out of {len(first_order_reviews):,}")

First orders with review text: 37,796 out of 93,658


### Loading the multilingual sentiment model
Using a pretrained BERT-based model trained on multilingual product reviews, 
since the review text here is in Portuguese. Testing on a small sample first 
to confirm it works and estimate how long the full run will take.

In [6]:
from transformers import pipeline
import time

# This downloads the model on first run (~600MB) — may take a few minutes
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
    max_length=256,
)

# Test on 20 reviews first, time it, before running on all 37,796
sample_reviews = first_order_reviews["review_comment_message"].dropna().sample(20, random_state=42).tolist()

start = time.time()
sample_results = sentiment_pipeline(sample_reviews)
elapsed = time.time() - start

print(f"Time for 20 reviews: {elapsed:.1f}s ({elapsed/20:.2f}s per review)")
print(f"\nEstimated time for full {n_with_text:,} reviews: {elapsed/20*n_with_text/60:.1f} minutes")

for text, result in zip(sample_reviews[:5], sample_results[:5]):
    print(f"\n{text[:80]}...")
    print(f"→ {result}")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

c:\Users\akiny\retention-uplift-project\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akiny\.cache\huggingface\hub\models--nlptown--bert-base-multilingual-uncased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Time for 20 reviews: 0.7s (0.04s per review)

Estimated time for full 37,796 reviews: 22.3 minutes

Recomendo esse produto! 
Ótima qualidade ...
→ {'label': '5 stars', 'score': 0.6924204230308533}

A única coisa q eu achei q tinha q ter vindo era o parafuso para instalar....mas...
→ {'label': '3 stars', 'score': 0.4308888614177704}

Comprei o produto pra receber no dia 31/01
E recebo ele dia 11/01 . 20 dias ant...
→ {'label': '1 star', 'score': 0.6154845952987671}

Produto perfeito e entrega muito rápida. 3 dias já estava comigo. ...
→ {'label': '5 stars', 'score': 0.8625342845916748}

Não recebi a mercadoria, mandei um e-mail para empresa pois estava com problema ...
→ {'label': '1 star', 'score': 0.6795300841331482}


### Running sentiment scoring on all first-order reviews
~22 minutes estimated. Converting the 1-5 star label into a numeric score so 
it can be used as a model feature, and keeping the confidence score too in 
case it's useful.


In [7]:
reviews_to_score = first_order_reviews.dropna(subset=["review_comment_message"]).copy()
texts = reviews_to_score["review_comment_message"].tolist()

print(f"Scoring {len(texts):,} reviews — this will take a few minutes...")
start = time.time()

# Process in batches for efficiency and to show progress
batch_size = 500
all_results = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    results = sentiment_pipeline(batch)
    all_results.extend(results)
    elapsed = time.time() - start
    print(f"  {i+len(batch):,}/{len(texts):,} done ({elapsed/60:.1f} min elapsed)", end="\r")

print(f"\nDone in {(time.time()-start)/60:.1f} minutes")

# Extract numeric star rating (1-5) and confidence score
reviews_to_score["sentiment_stars"] = [int(r["label"][0]) for r in all_results]
reviews_to_score["sentiment_confidence"] = [r["score"] for r in all_results]

print(reviews_to_score[["review_comment_message", "sentiment_stars", "sentiment_confidence"]].head())

Scoring 37,796 reviews — this will take a few minutes...
  37,796/37,796 done (19.5 min elapsed)
Done in 19.5 minutes
                               review_comment_message  sentiment_stars  \
0   Adorei a cortina, ficou linda na minha sala, e...                5   
3                                        Bom vendedor                5   
7   Olá! Comprei dois potes de whey e chegou apena...                1   
9   Até o presente momento não recebi o produto e ...                1   
10    só achei que a altura da saia poderia ser maior                3   

    sentiment_confidence  
0               0.842027  
3               0.534693  
7               0.830599  
9               0.597157  
10              0.522656  


### Saving sentiment scores
Saving separately from the main feature table so this expensive step doesn't 
need to be re-run if anything downstream changes.

In [8]:
output_cols = ["customer_unique_id", "sentiment_stars", "sentiment_confidence"]
sentiment_output = reviews_to_score[output_cols]
sentiment_output.to_csv("../data/processed/review_sentiment.csv", index=False)
print(f"Saved {len(sentiment_output):,} sentiment scores")

Saved 37,796 sentiment scores


### Merging sentiment into the model table
Left-joining sentiment scores onto the full feature table. Most customers won't 
have a sentiment score (only ~40% left review text), so we need a flag for 
"no review text" separate from imputing a neutral score — same pattern used for 
review_score back in feature engineering.

In [9]:
sentiment_df = pd.read_csv("../data/processed/review_sentiment.csv")
model_data_v2 = model_data.merge(sentiment_df, on="customer_unique_id", how="left")

model_data_v2["has_review_text"] = model_data_v2["sentiment_stars"].notna().astype(int)
model_data_v2["sentiment_stars"] = model_data_v2["sentiment_stars"].fillna(
    model_data_v2["sentiment_stars"].median()
)
model_data_v2["sentiment_confidence"] = model_data_v2["sentiment_confidence"].fillna(0)

print(f"Shape after merge: {model_data_v2.shape}")
print(f"Customers with review text: {model_data_v2['has_review_text'].sum():,}")

model_data_v2.to_csv("../data/processed/model_ready_v2.csv", index=False)

Shape after merge: (55960, 46)
Customers with review text: 23,095


### Re-running XGBoost with the sentiment feature added
Same setup as Step H, but with sentiment_stars, sentiment_confidence, and 
has_review_text added. If performance improves meaningfully, sentiment adds 
real signal beyond the star rating we already had. If not, that's still a 
useful, honest finding.

In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

model_data_v2["first_purchase_date"] = pd.to_datetime(model_data_v2["first_purchase_date"])
model_data_v2 = model_data_v2.sort_values("first_purchase_date").reset_index(drop=True)

split_date_v2 = model_data_v2["first_purchase_date"].quantile(0.8)
train_v2 = model_data_v2[model_data_v2["first_purchase_date"] < split_date_v2]
test_v2 = model_data_v2[model_data_v2["first_purchase_date"] >= split_date_v2]

drop_cols_v2 = ["customer_unique_id", "first_purchase_date", "made_second_purchase"]
X_train_v2 = train_v2.drop(columns=drop_cols_v2)
y_train_v2 = train_v2["made_second_purchase"]
X_test_v2 = test_v2.drop(columns=drop_cols_v2)
y_test_v2 = test_v2["made_second_purchase"]

neg2, pos2 = y_train_v2.value_counts()
xgb_v2 = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=neg2/pos2, eval_metric="aucpr", random_state=42,
)
xgb_v2.fit(X_train_v2, y_train_v2)

y_proba_v2 = xgb_v2.predict_proba(X_test_v2)[:, 1]

# Step H baseline (from previous notebook) — hardcoded since notebooks
# don't share variables with each other
baseline_roc_auc = 0.5727
baseline_pr_auc = 0.0468

print("=== WITHOUT sentiment (Step H baseline) ===")
print(f"ROC-AUC: {baseline_roc_auc:.4f}, PR-AUC: {baseline_pr_auc:.4f}")

print("\n=== WITH sentiment ===")
print(f"ROC-AUC: {roc_auc_score(y_test_v2, y_proba_v2):.4f}, PR-AUC: {average_precision_score(y_test_v2, y_proba_v2):.4f}")

=== WITHOUT sentiment (Step H baseline) ===
ROC-AUC: 0.5727, PR-AUC: 0.0468

=== WITH sentiment ===
ROC-AUC: 0.5678, PR-AUC: 0.0467
